In [ ]:
import gc
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from pandas.tseries.holiday import USFederalHolidayCalendar

warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

PATH = "../../data/ieee"  # thư mục chứa train_transaction.csv, train_identity.csv
SEED = 42
TARGET = "isFraud"
START_DATE = pd.Timestamp("2017-11-30")

PARAMS = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "metric": "auc",
    "n_jobs": -1,
    "learning_rate": 0.01,
    "num_leaves": 2**8,
    "max_depth": -1,
    "tree_learner": "serial",
    "colsample_bytree": 0.7,
    "subsample_freq": 1,
    "subsample": 0.7,
    "n_estimators": 80000,
    "max_bin": 255,
    "verbose": -1,
    "seed": SEED,
    "early_stopping_rounds": 100,
}

In [2]:
# ---------- Hàm phụ ----------
def freq(s):
    # số lần xuất hiện của từng giá trị (NaN tính như 1 giá trị)
    codes = pd.factorize(s)[0] + 1
    return np.bincount(codes)[codes]


def group_agg(df, keys, cols, aggs=("mean", "std")):
    # với mỗi key: mean/std của cols theo nhóm key
    new = {}
    for k in keys:
        g = df.groupby(k)[cols]
        for a in aggs:
            r = g.transform(a)
            for c in cols:
                new[f"{k}_{c}_{a}"] = r[c].to_numpy(dtype="float32")
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)


def normalize(df, cols, periods):
    # min-max và z-score của cols trong từng khối thời gian
    x = df[cols].astype("float32")
    new = []
    for p in periods:
        g = x.groupby(df[p])
        mn, mx, mean, std = (
            g.transform("min"),
            g.transform("max"),
            g.transform("mean"),
            g.transform("std"),
        )
        new.append(((x - mn) / (mx - mn)).add_suffix(f"_{p}_min_max"))
        new.append(((x - mean) / std).add_suffix(f"_{p}_std_score"))
    return pd.concat([df] + new, axis=1)

In [3]:
# ---------- Load + chia train/test theo tháng ----------
df = pd.read_csv(f"{PATH}/train_transaction.csv").merge(
    pd.read_csv(f"{PATH}/train_identity.csv"), on="TransactionID", how="left"
)
f64 = df.select_dtypes("float64").columns
df[f64] = df[f64].astype("float32")

dt = START_DATE + pd.to_timedelta(df["TransactionDT"], unit="s")
df["DT_M"] = ((dt.dt.year - 2017) * 12 + dt.dt.month).astype("int8")

last = df["DT_M"].max()
df = df[df["DT_M"] != last - 1].reset_index(drop=True)  # bỏ tháng sát cuối
is_train = (df["DT_M"] < last).to_numpy()  # False = tháng test
print("train:", is_train.sum(), "| test:", (~is_train).sum())

train: 417559 | test: 89326


In [4]:
# ---------- 1. Đặc trưng thời gian ----------
dt = START_DATE + pd.to_timedelta(df["TransactionDT"], unit="s")
holidays = USFederalHolidayCalendar().holidays(start="2017-10-01", end="2019-01-01")

df["DT_D"] = (df["TransactionDT"] // 86400).astype("int16")  # ngày
df["DT_W"] = (df["DT_D"] // 7).astype("int16")  # tuần
df["DT_hour"] = dt.dt.hour.astype("int8")
df["DT_day_week"] = dt.dt.dayofweek.astype("int8")
df["DT_day_month"] = dt.dt.day.astype("int8")
df["is_december"] = (dt.dt.month == 12).astype("int8")
df["is_holiday"] = dt.dt.normalize().isin(holidays).astype("int8")

# tổng số giao dịch mỗi khối thời gian (chỉ dùng để chia, không đưa vào model)
for p in ["DT_M", "DT_W", "DT_D"]:
    df[p + "_total"] = df[p].map(df[p].value_counts())

In [5]:
# ---------- 2. Làm sạch card + tạo uid ----------
# card1: bỏ giá trị hiếm (<= 2 lần)
df["card1"] = df["card1"].where(df["card1"].map(df["card1"].value_counts()) > 2)

# card1..card6: chỉ giữ giá trị có mặt ở cả train và test
for c in ["card1", "card2", "card3", "card4", "card5", "card6"]:
    common = set(df.loc[is_train, c].dropna()) & set(df.loc[~is_train, c].dropna())
    df[c] = df[c].where(df[c].isin(common))

s = lambda c: df[c].astype(str)
df["uid"] = s("card1") + "_" + s("card2")
df["uid2"] = df["uid"] + "_" + s("card3") + "_" + s("card5")
df["uid3"] = df["uid2"] + "_" + s("addr1") + "_" + s("addr2")
df["uid4"] = df["uid3"] + "_" + s("P_emaildomain")
df["uid5"] = df["uid3"] + "_" + s("R_emaildomain")
df["bank_type"] = s("card3") + "_" + s("card5")
uids = ["uid", "uid2", "uid3", "uid4", "uid5"]

for c in ["card1", "card2", "card3", "card5"] + uids:
    df[c + "_fq"] = freq(df[c])

In [6]:
# ---------- 3. bank_type: giờ/thứ/ngày giao dịch lệch bao nhiêu so với trung bình nhóm ----------
new = {}
for c in ["card3", "card5", "bank_type"]:
    key = df[c].astype(str)
    for p, t in [
        ("DT_D", "DT_hour"),
        ("DT_W", "DT_day_week"),
        ("DT_M", "DT_day_month"),
    ]:
        new[f"{c}_{p}_{t}_dist"] = df[t] - df.groupby([key, df[p]])[t].transform("mean")

# tỉ lệ giao dịch của bank_type trong từng khối thời gian
for p in ["DT_M", "DT_W", "DT_D"]:
    new[f"bank_type_{p}_prop"] = (
        df.groupby(["bank_type", p])["TransactionID"].transform("size")
        / df[p + "_total"]
    )

df = pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

In [7]:
# ---------- 4. Cột D ----------
D = [f"D{i}" for i in range(1, 16)]
periods = ["DT_D", "DT_W", "DT_M"]

df = group_agg(df, uids + ["bank_type"], D)  # mean/std theo uid (trên giá trị gốc)

df[D] = df[D].clip(lower=0)  # timedelta không thể âm
df["D9_not_na"] = df["D9"].notna().astype("int8")
df["D8_not_same_day"] = (df["D8"] >= 1).astype("int8")
df["D8_D9_decimal_dist"] = ((df["D8"].fillna(0) % 1) - df["D9"]).abs()
df["D8"] = df["D8"].fillna(-1).astype(int)

df = normalize(df, [c for c in D if c not in ("D1", "D2", "D9")], periods)
for c in ["D1", "D2"]:
    df[c + "_scaled"] = df[c] / df.loc[is_train, c].max()
for c in D:
    df[c] = freq(df[c])  # thay giá trị D bằng tần suất

In [8]:
# ---------- 5. TransactionAmt ----------
df["TransactionAmt"] = df["TransactionAmt"].clip(0, 5000)

# số tiền này có xuất hiện ở phía bên kia (train <-> test) không
amt_train, amt_test = (
    set(df.loc[is_train, "TransactionAmt"]),
    set(df.loc[~is_train, "TransactionAmt"]),
)
df["TransactionAmt_check"] = np.where(
    is_train, df["TransactionAmt"].isin(amt_test), df["TransactionAmt"].isin(amt_train)
).astype("int8")

df = group_agg(
    df, ["card1", "card2", "card3", "card5"] + uids + ["bank_type"], ["TransactionAmt"]
)
df = normalize(df, ["TransactionAmt"], periods)

# ProductCD + số tiền
df["product_type"] = (
    df["ProductCD"].astype(str) + "_" + df["TransactionAmt"].astype(str)
)
for p in ["DT_M", "DT_W", "DT_D"]:
    df[f"product_type_{p}_prop"] = (
        df.groupby(["product_type", p])["TransactionID"].transform("size")
        / df[p + "_total"]
    )
df["product_type"] = freq(df["product_type"])

df["TransactionAmt"] = np.log1p(df["TransactionAmt"])

In [9]:
C = [f"C{i}" for i in range(1, 15)]
for c in C:
    df[c + "_fq"] = freq(df[c])


cap = df.loc[df["DT_M"] == df.loc[is_train, "DT_M"].max(), C].max()
df[C] = df[C].clip(upper=cap, axis=1)

In [10]:
# ---------- 7. Identity: DeviceInfo, id_30, id_31 ----------
for c in ["DeviceInfo", "id_30", "id_31"]:
    df[c] = df[c].fillna("unknown_device").str.lower()

df["DeviceInfo_device"] = df["DeviceInfo"].str.replace(r"[^a-z]", "", regex=True)
df["DeviceInfo_version"] = df["DeviceInfo"].str.replace(r"\D", "", regex=True)
df["id_30_device"] = df["id_30"].str.replace(r"[^a-z]", "", regex=True)
df["id_30_version"] = df["id_30"].str.replace(r"\D", "", regex=True)
df["id_31_device"] = df["id_31"].str.replace(r"[^a-z]", "", regex=True)

for c in [
    "DeviceInfo",
    "DeviceInfo_device",
    "DeviceInfo_version",
    "id_30",
    "id_30_device",
    "id_30_version",
    "id_31",
    "id_31_device",
    "id_33",
]:
    df[c] = freq(df[c])

# ---------- 8. Target mean (chỉ tính trên train) ----------
for c in ["ProductCD", "M4"]:
    df[c] = df[c].map(df.loc[is_train].groupby(c)[TARGET].mean())

In [11]:
# ---------- Chọn feature + chuẩn bị cho LightGBM ----------
drop = [
    "TransactionID",
    "TransactionDT",
    TARGET,
    "DT_M",
    "DT_W",
    "DT_D",
    "DT_hour",
    "DT_day_week",
    "DT_day_month",
    "DT_M_total",
    "DT_W_total",
    "DT_D_total",
    "bank_type",
] + uids
features = [c for c in df.columns if c not in drop]

for c in df[features].select_dtypes(exclude="number").columns:
    df[c] = df[c].astype("category")

num = df[features].select_dtypes("float").columns
df[num] = df[num].replace([np.inf, -np.inf], np.nan).astype("float32")
gc.collect()

train, test = df[is_train], df[~is_train]
print("train:", train.shape, "| test:", test.shape, "| số feature:", len(features))

train: (417559, 779) | test: (89326, 779) | số feature: 761


In [ ]:
dtrain = lgb.Dataset(train[features], train[TARGET])
dtest = lgb.Dataset(test[features], test[TARGET])

# lưu ý: early stopping dựa trên chính tháng test -> AUC hơi lạc quan,
# nhưng file 02 làm y hệt nên so sánh vẫn công bằng
model = lgb.train(
    PARAMS,
    dtrain,
    num_boost_round=3000,
    valid_sets=[dtest],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

pred = model.predict(test[features], num_iteration=model.best_iteration)
print("AUC:", roc_auc_score(test[TARGET], pred))

Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.891942
[200]	valid_0's auc: 0.905617
[300]	valid_0's auc: 0.913896
[400]	valid_0's auc: 0.918433
[500]	valid_0's auc: 0.920588
[600]	valid_0's auc: 0.921862
[700]	valid_0's auc: 0.922596
[800]	valid_0's auc: 0.923017
[900]	valid_0's auc: 0.922973
Early stopping, best iteration is:
[840]	valid_0's auc: 0.923141


In [13]:
imp = pd.Series(model.feature_importance("gain"), index=features).sort_values(
    ascending=False
)
imp.head(30)

V258                           70934.236844
C1_fq                          25458.302394
C14                            23504.521854
V294                           16789.984362
C13                            14158.916673
V243                           13831.427796
C1                             13396.696845
V257                           12125.402938
R_emaildomain                  12027.757056
D2_scaled                      10407.435735
V70                             7058.709536
V317                            6935.139545
addr1                           6598.692671
product_type_DT_W_prop          6439.799076
C8                              6302.425789
card2_TransactionAmt_mean       6217.604891
card1_TransactionAmt_std        6214.773975
P_emaildomain                   6205.749496
C7                              6194.245617
C14_fq                          5656.681067
card1                           5646.296650
V149                            5588.633634
card1_TransactionAmt_mean       

In [14]:
from util import summary

summary(test[TARGET], pred)

accuracy_score 0.9757405458657054
recall_score: 0.3930635838150289
precision_score: 0.8154563624250499
f1_score 0.5304442036836403
ROC-AUC score: 0.9188978247555182
PRAUC score: 0.5783616824893161
COnfusion matrix: [[85935   277]
 [ 1890  1224]]
